In [0]:
# 06_export_site_charts.ipynb
import os
import json
from datetime import datetime, timezone

site_root = "/Workspace/Users/john.bain@ssc-spc.gc.ca/Artemis 2/fsdh-artemis-demo"
data_dir = os.path.join(site_root, "data")

os.makedirs(data_dir, exist_ok=True)

print(f"Site root ready: {site_root}")
print(f"Data directory ready: {data_dir}")

speed_df = (
    spark.table("fsdh_artemis.vw_orion_speed_over_time")
    .orderBy("epoch_utc")
    .toPandas()
)

distance_df = (
    spark.table("fsdh_artemis.vw_orion_distance_over_time")
    .orderBy("epoch_utc")
    .toPandas()
)

latest_kpis_df = spark.table("fsdh_artemis.vw_orion_latest_kpis")
latest_kpis_row = latest_kpis_df.first()

metadata_df = spark.table("fsdh_artemis.vw_orion_metadata_wide")
metadata_row = metadata_df.first()

speed_df["speed_km_h"] = speed_df["speed_km_s"] * 3600.0

display(latest_kpis_df)
display(metadata_df)

In [0]:
def iso_or_blank(value):
    if value is None:
        return ""
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)

speed_points = [
    {
        "epoch_utc": iso_or_blank(row["epoch_utc"]),
        "x": float(row["mission_elapsed_hours"]),
        "y": float(row["speed_km_h"])
    }
    for _, row in speed_df.iterrows()
]

distance_points = [
    {
        "epoch_utc": iso_or_blank(row["epoch_utc"]),
        "x": float(row["mission_elapsed_hours"]),
        "y": float(row["distance_from_origin_km"])
    }
    for _, row in distance_df.iterrows()
]

print(f"Speed points: {len(speed_points)}")
print(f"Distance points: {len(distance_points)}")

print("First speed point:")
print(speed_points[0] if speed_points else "None")

print("First distance point:")
print(distance_points[0] if distance_points else "None")

In [0]:
mission_data_path = os.path.join(data_dir, "mission-data.json")

latest_epoch_utc = str(latest_kpis_row["latest_epoch_utc"])
latest_speed_km_s = float(latest_kpis_row["speed_km_s"])
latest_speed_km_h = latest_speed_km_s * 3600.0

state_vector_count = len(speed_df)
mission_duration_hours = float(speed_df["mission_elapsed_hours"].max())

mission_data = {
    "generated_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "mission": {
        "title": "Artemis II Mission Overview",
        "latest_epoch_utc": latest_epoch_utc,
        "latest_speed_km_s": latest_speed_km_s,
        "latest_speed_km_h": latest_speed_km_h,
        "state_vector_count": state_vector_count,
        "mission_duration_hours": mission_duration_hours
    },
    "metadata": {
        "object_name": metadata_row["object_name"] if metadata_row and metadata_row["object_name"] is not None else "",
        "object_id": metadata_row["object_id"] if metadata_row and metadata_row["object_id"] is not None else "",
        "center_name": metadata_row["center_name"] if metadata_row and metadata_row["center_name"] is not None else "",
        "reference_frame": metadata_row["ref_frame"] if metadata_row and metadata_row["ref_frame"] is not None else "",
        "time_system": metadata_row["time_system"] if metadata_row and metadata_row["time_system"] is not None else "",
        "start_time": str(metadata_row["start_time"]) if metadata_row and metadata_row["start_time"] is not None else "",
        "stop_time": str(metadata_row["stop_time"]) if metadata_row and metadata_row["stop_time"] is not None else ""
    },
    "charts": {
        "speed": {
            "x_label_key": "missionElapsedHours",
            "y_label_key": "speedKilometresPerHour",
            "points": speed_points
        },
        "distance": {
            "x_label_key": "missionElapsedHours",
            "y_label_key": "distanceFromOriginKilometres",
            "points": distance_points
        }
    }
}

with open(mission_data_path, "w", encoding="utf-8") as f:
    json.dump(mission_data, f, indent=2)

print(f"Saved: {mission_data_path}")

print("Data directory contents:")
for name in sorted(os.listdir(data_dir)):
    print("-", name)